# `subvillage` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `high-cardinality-category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'subvillage'
feature_metadata = {'order': 11, 'name': 'subvillage', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'exclude raw one-hot value from the first baseline', 'finding': 'The field is extremely sparse, has reused names and exposes many test rows to unseen levels.', 'decision': 'Use only a separately validated hashing or frequency treatment with explicit missingness.', 'risk': 'Direct encoding encourages local memorisation and poor unseen coverage.', 'related': [{'feature': 'ward', 'reason': 'Subvillages sit within wards, although names are reused.'}, {'feature': 'lga', 'reason': 'LGA context partially disambiguates repeated subvillage names.'}, {'feature': 'wpt_name', 'reason': 'Waterpoint names can repeat within local settlements.'}, {'feature': 'region', 'reason': 'Region is the broad administrative back-off.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for subvillage.


## Supported target evidence


In [2]:
sentinel_tokens = []
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
subvillage,,,,,
madukani,508,True,48.82,8.46,42.72
shuleni,506,True,45.65,8.70,45.65
majengo,502,True,46.61,7.17,46.22
kati,373,True,57.64,11.53,30.83
<missing/blank>,371,True,55.26,0.27,44.47
mtakuja,262,True,54.20,5.34,40.46
sokoni,232,True,47.84,5.60,46.55
m,187,True,66.31,1.60,32.09
muungano,172,True,45.93,7.56,46.51


status_group,rows,non functional (%)
subvillage,,
1,132,54.55
mapinduzi,109,54.13
mwenge,102,50.00
miembeni,134,48.51
msikitini,134,48.51
sokoni,232,46.55
muungano,172,46.51
majengo,502,46.22
shuleni,506,45.65


## Observation

The field is extremely sparse, has reused names and exposes many test rows to unseen levels.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Use only a separately validated hashing or frequency treatment with explicit missingness.

**Risk to carry forward:** Direct encoding encourages local memorisation and poor unseen coverage.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
subvillage,candidate,exclude raw one-hot value from the first baseline,"The field is extremely sparse, has reused name...",Use only a separately validated hashing or fre...,Direct encoding encourages local memorisation ...
